# Donut fine-tune - InBody **270 + 570** v5 - Kaggle runner

Retrains Donut on the **v5** both-device synthetic set (2500 inbody_270 + 2500 inbody_570).

**What v5 changes, and it is one hypothesis.** The Segmental Fat panel taught the model a cue
that exists only in training. On the 270 our figure printed two lines per limb (kg and a status
hardcoded to `Normal`) while Segmental Lean printed three, so three lines meant Lean and two
meant Fat and the headings never had to be read. A real 270 prints three in both. On the 570 the
same commit skewed the left/right segmental fat values apart; they used to be the same number on
every sheet, which is its own training-only cue (23% of 570 sheets changed as a side effect -
see ADR-0007's 2026-09-11 amendment, which is where that was found and written up).

Both halves push the same way: remove a Segmental Fat cue present in training and absent at
inference. That is the hypothesis this run tests. See issue #50 and ADR-0007.

**Not** the geometry hypothesis - that was v4, and it was not supported (`docs/ocr-eval-results.md`).

Fresh from `naver-clova-ix/donut-base`, **3 epochs**, **both devices**. Do **not** resume from
`donut-both-v3` or any v4 checkpoint - the input distribution changed again.

**Kaggle setup before running:**
1. *Settings -> Accelerator* -> **GPU T4 x2** (we pin to one GPU below - donut-base OOMs on GPU0 with both).
2. *Settings -> Internet* -> **On**.
3. *Add-ons -> Secrets* -> add **`GH_TOKEN`** (fine-grained, Contents: Read-only, scoped to QeekOw/InForm).
4. Upload `synth_v5_both.zip` to **Kaggle -> Datasets -> New Dataset**, then attach it here via
   **Add Input** (right panel). Cell 3 auto-detects + extracts it.
5. The branch below must be **pushed** - this notebook trains the code it clones, not your
   working tree.

**Why 3 epochs and not 5.** v4 asked for 5, was killed by the 12 h cap at epoch 4, and never
wrote epoch 5. Measured ~3 h/epoch on 5000 sheets, so 3 epochs finishes inside the cap with
room. Two reasons that is the right call rather than a concession: v4's **best** checkpoint was
epoch 3 (30/36 core) and epoch 4 was much worse (13/36), so there is no evidence more epochs
help on this data; and `train.py` saves the processor only on a **completed** run, so a run cut
short leaves checkpoints that cannot be loaded without rebuilding the processor by hand. If the
result looks underfit rather than mis-taught, raise it to 4 and expect to use the resume cell.


In [ ]:
# GPU + the two flags from prior runs: pin to one GPU (dual-T4 OOMs donut-base on
# GPU0) and enable expandable segments to avoid fragmentation OOMs.
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Clone the branch carrying the v5 templates, the manifest and the silent-error scorer.
BRANCH = 'feat/module-1-real-holdout-scorer'
from kaggle_secrets import UserSecretsClient
try:
    GH_TOKEN = UserSecretsClient().get_secret('GH_TOKEN')
    REPO = f'https://{GH_TOKEN}@github.com/QeekOw/InForm.git'
except Exception:
    REPO = 'https://github.com/QeekOw/InForm.git'  # public fallback
%cd /kaggle/working
!rm -rf /kaggle/working/repo
!git clone --branch $BRANCH --single-branch $REPO repo
%cd /kaggle/working/repo
!pip install -q -e '.[training]' gdown

In [ ]:
# Dataset attached as a Kaggle Dataset (Add Input, right panel) - no Drive/gdown.
# Find the sheets wherever Kaggle mounted them; if the input is still a .zip,
# extract it to /kaggle/tmp. Sets DATA_DIR to the folder holding the sheets.
import glob, os, shutil

# generate_dataset writes .jpg (inform.training.dataset.IMAGE_SUFFIXES); runs
# before that wrote .png, so accept either rather than silently finding nothing.
SUFFIXES = ('jpg', 'jpeg', 'png')

def find_sheets(root):
    return sorted(p for s in SUFFIXES for p in glob.glob(f'{root}/**/*.{s}', recursive=True))

sheets = find_sheets('/kaggle/input')
if not sheets:
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    assert zips, 'No sheets or zip under /kaggle/input - click Add Input and attach your dataset.'
    shutil.unpack_archive(zips[0], '/kaggle/tmp')
    sheets = find_sheets('/kaggle/tmp')
assert sheets, 'No sheets found after extract - check the dataset contents.'
DATA_DIR = os.path.dirname(sheets[0])
n270 = len([p for p in sheets if 'inbody_270' in p]); n570 = len([p for p in sheets if 'inbody_570' in p])
print('DATA_DIR =', DATA_DIR, '| sheets:', len(sheets), '| 270:', n270, '570:', n570)


In [ ]:
# Fresh from donut-base, 3 epochs, BOTH devices. Do NOT resume from v3 or v4:
# v5 changed the input distribution again (Segmental Fat panel on both devices).
# batch 1 + grad-accum 4 fits the 2560x1920 canvas on a T4; effective batch 4.
# --batch-size 2 OOMs even on 16 GB - raise grad-accum instead.
# 4 dataloader workers keep the GPU fed while JPEGs decode.
# 3 epochs fits the 12 h cap, so the run completes and saves its processor.
CHECKPOINT_DIR = '/kaggle/working/donut-both-v5'
!python -m inform.training.train \
  --data-dir "$DATA_DIR" --output-dir $CHECKPOINT_DIR \
  --model-name-or-path naver-clova-ix/donut-base \
  --epochs 3 --batch-size 1 --gradient-accumulation-steps 4 \
  --dataloader-num-workers 4 --learning-rate 3e-5


In [ ]:
# RESUME ONLY - run this instead of cell 4 when a previous session hit the 12 h cap.
# Attach that session's output as an input, copy the checkpoints into CHECKPOINT_DIR,
# then --resume picks up from the last checkpoint-* subdir.
#
# import glob, shutil, os
# prev = glob.glob('/kaggle/input/**/donut-both-v5', recursive=True)[0]
# shutil.copytree(prev, CHECKPOINT_DIR, dirs_exist_ok=True)
# !python -m inform.training.train \
#   --data-dir "$DATA_DIR" --output-dir $CHECKPOINT_DIR \
#   --model-name-or-path naver-clova-ix/donut-base \
#   --epochs 3 --batch-size 1 --gradient-accumulation-steps 4 \
#   --dataloader-num-workers 4 --learning-rate 3e-5 --resume


In [ ]:
# Every epoch checkpoint is kept (save_strategy='epoch', save_total_limit=None), so
# /kaggle/working holds checkpoint-* subdirs plus the final top-level model. All of it
# lands in the Save Version output. ~2.4 GB per checkpoint - watch the 20 GB /kaggle/working
# limit, and download to a drive with room (NOT C:, which has ~12 GB free).
!du -sh /kaggle/working/donut-both-v5/* | sort -h
!ls -la /kaggle/working/donut-both-v5


## After the run

Score **every** epoch checkpoint, not just the last.

### The headline is the silent-error rate

A silent error is a wrong value inside an `unverified` read: wrong, and carrying no signal that
anything is wrong (CONTEXT.md, and ADR-0006's 2026-09-11 amendment). Per-field accuracy is a
diagnostic beneath it.

```bash
python -m inform.holdout --data-dir data/real_holdout \
    --labels data/real_holdout/labels.json --donut-checkpoint <ckpt>
```

Baseline to beat (`donut-both-v3`, measured 2026-09-11 under the current parser):

| | v3 |
|---|---|
| **unverified sheets carrying >=1 wrong value** | **3/3 (100%)** |
| their fields, wrong | 4/33 (12.1%) |
| unverified reads with no hand label | 0 (not measurable) |
| core fields | 33/36 (91.7%) |
| segmental lean | 23/30 (76.7%) |
| critical (LBM + limbs) | 26/36 (72.2%) |
| outcome split (n=12) | 3 unverified / 6 flagged / 3 unread / 0 refused |

**n = 3 on the headline.** It is a floor on one checkpoint, not a basis for ranking two. Six of
twelve sheets carry hand labels at all (#24), and that denominator is the binding constraint on
this whole exercise - not the model.

Earlier notebooks quote v3's split as `3 usable / 3 flagged / 0 unread / 6 refused`. That is the
stale figure: it came from replaying a recorded-reads file written before the malformed-
generation parser fix. Replay `donut-both-v3-reads-57aa760.json`, not `donut-both-v3-reads.json`.

### The sharpest single indicator is not an aggregate

Check the **sheet_05 panel crossing**. That sheet reads `unverified` with every cross-check
passing while its right-arm and right-leg lean values are the adjacent Segmental Fat panel's
left-side values. It survived v4. If the panel cue was the cause, that specific failure goes
away; `segmental_lean.right_arm_kg` (1/6 under v3, 2/6 under v4 epoch 3) is the field to watch.

### Held-out synthetic (both devices)

```bash
python -m inform.compare --data-dir D:/cera/data/holdout_v5_both \
    --donut-checkpoint <ckpt> --skip-vlm
```

Use `holdout_v5_both`, not `holdout_v4_both`: the 570 half of the v4 sets is not reproducible
from this commit, so scoring v5 weights against v4 sheets compares across two generators.
Synthetic 570 numbers validate the extractor, not the synthetic-to-real transfer - there is
still no real metric 570 photo (#24).

### Check the dataset is the one you think

Each shard of a generated set writes a `dataset.*.json` manifest naming its devices, seed range
and a fingerprint over the generator and both templates. Confirm the fingerprints agree across
shards and match the commit you trained, before believing any comparison between two runs.
